# Model Selection: ARMA

<a href="https://colab.research.google.com/github/febse/ts2025/blob/main/05-ARMA-Fitting-Class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Class Version In Colab"/>Class Version</a>


In [ ]:
# Install missing packages (for Google Colab compatibility) and load them
required_packages <- c("tidyverse", "forecast", "patchwork")
missing_packages <- required_packages[!required_packages %in% rownames(installed.packages())]
if (length(missing_packages) > 0) install.packages(missing_packages)

library(tidyverse)
library(forecast)
library(patchwork)


In [ ]:
# A purely random process with a standard deviation of 1

# AR(1) parameters
phi_1 <- 0.8   # AR coefficient
phi_2 <- -0.6
theta_1 <- 0.6 # MA coefficient
theta_2 <- -0.4
sigma <- 1     # standard deviation of noise
n <- 100

set.seed(42)

noise <- arima.sim(model = list(order = c(0, 0, 0)), n = n, sd = sigma)

dt <- arima.sim(model = list(ar = phi_1), n = n, sd = 1.5 * sigma)

dt_ex <- read_csv("https://github.com/febse/data/raw/refs/heads/main/ts/ts2025/a1/ts27.csv") |> pull(1)


## Fitting AR(p) Models

$$
x_t = \phi_0 + \phi_1 x_{t-1} + \phi_2 x_{t-2} + ... + \phi_p x_{t-p} + e_t
$$

where $e_t \sim N(0, \sigma_e^2), E(e_t e_{t-k}) = 0 \text{ for } k \neq 0$.

In [ ]:
# Plot the series
dt_df <- tibble(time = seq_along(dt), value = as.numeric(dt))

ggplot(dt_df, aes(x = time, y = value)) +
  geom_line() +
  labs(title = "Time Series Plot", x = "Time", y = "Value")


In [ ]:
# Plot ACF and PACF
p_acf <- ggAcf(dt, lag.max = 20) + labs(title = "ACF")
p_pacf <- ggPacf(dt, lag.max = 20) + labs(title = "PACF")

p_acf + p_pacf


In [ ]:
# Fit an AR(1) model and print the summary
model_ar1_fit <- Arima(dt, order = c(1, 0, 0))
summary(model_ar1_fit)


In [ ]:
# Residual diagnostics (standardized residuals, ACF, histogram) and Ljung-Box test
checkresiduals(model_ar1_fit)


## The Jung-Box Test

$$
Q=T(T+2)\sum _{k=1}^{h}{\frac {{\hat {\rho }}_{k}^{2}}{T-k}} \underset{H_0}{\sim} \chi(h)
$$

$$
\begin{align*}
H_0: & \rho_1 = \rho_2 = \ldots = \rho_{h} \\
H_1: & \text{ at least one of } \rho_k \neq 0, \quad k = 1,\ldots, h
\end{align*}
$$

In [ ]:
# Ljung-Box test of residual autocorrelation at multiple lags
for (h in 1:10) {
  lb <- Box.test(residuals(model_ar1_fit), lag = h, type = "Ljung-Box")
  cat(sprintf("Lag %d, test statistic = %.3f with p-value %.3f\n", h, lb$statistic, lb$p.value))
}


In [ ]:
fc <- predict(model_ar1_fit, n.ahead = 2)

cat("Forecasts\n")
print(fc$pred)

cat("Standard errors\n")
print(fc$se)


In [ ]:
# Plot the prediction of the model

fitted_values <- fitted(model_ar1_fit)

forecast_steps <- 20
fc <- forecast(model_ar1_fit, h = forecast_steps)

forecast_index <- (length(dt) + 1):(length(dt) + forecast_steps)

plot_df <- tibble(
  time = seq_along(dt),
  original = as.numeric(dt),
  fitted = as.numeric(fitted_values)
)

forecast_df <- tibble(
  time = forecast_index,
  forecast = as.numeric(fc$mean),
  lower95 = as.numeric(fc$lower[, "95%"]),
  upper95 = as.numeric(fc$upper[, "95%"])
)

ggplot() +
  geom_line(data = plot_df, aes(x = time, y = original, color = "Original Data"), linewidth = 0.7) +
  geom_line(data = plot_df, aes(x = time, y = fitted, color = "Fitted Values"), linetype = "dashed", linewidth = 0.7) +
  geom_line(data = forecast_df, aes(x = time, y = forecast, color = "Forecast"), linewidth = 0.7) +
  geom_ribbon(data = forecast_df, aes(x = time, ymin = lower95, ymax = upper95), fill = "red", alpha = 0.2) +
  geom_vline(xintercept = length(dt), linetype = "dotted", color = "black") +
  scale_color_manual(values = c("Original Data" = "blue", "Fitted Values" = "green4", "Forecast" = "red")) +
  labs(title = "ARIMA(1,0,0) Model: Fitted Values and Forecasts", x = "Time", y = "Value", color = NULL) +
  theme_minimal()


## Information Criteria

### Akaike Information Criterion (AIC)

$$
AIC = -2 \underbrace{\log(L)}_{\text{Goodness of fit}} + 2 \underbrace{(p + q + k + 1)}_{\text{number of parameters}}
$$

Information criteria are designed to balance between goodness of fit (how close the model is to the data) and model complexity (how many parameters are in the model). Lower values of AIC indicate a better model.


$$
AICc = AIC + \frac{2(p + q + k + 1)(p + q + k + 2)}{T - p - q - k - 2}
$$

$$
BIC = AIC + (\log(T) - 2)(p + q + k + 1)
$$



## Maximum Likelihood Estimation (MLE)

Although it is outside of the scope for this course, it is worth mentioning that the parameters of ARMA models are typically estimated using Maximum Likelihood Estimation (MLE) or variants thereof (e.g., Conditional MLE). But what is maximum likelihood estimation? This is a rather general method for estimating parameters of a statistical model. Maybe the simplest example is to show it for estimating the probability of success in a binomial distribution.

Let's say that we have an experiment where each outcome is either a success or a failure. We denote the probability of success as $p$ and the probability of failure as $1-p$. If we repeat this experiment $n$ times, the number of successes $k$ follows a binomial distribution:

$$
P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}
$$

Now let's say that we interview 10 persons and ask them whether they support a certain policy. Let's say that 7 out of the 10 persons say "yes". Let the event persons supports the policy be a success. Then we have $n=10$ and $k=7$. Now we can ask the question: What is the value of $p$ that makes the observed data most likely? In other words, what is the value of $p$ that maximizes the probability of observing 7 successes in 10 trials?

Assuming each person answers independently from the others, the likelihood function is given by:

$$
L(p) = P(X = 7) = \binom{10}{7} p^7 (1-p)^{3}
$$

To find the value of $p$ that maximizes this likelihood function, we can take the derivative of $L(p)$ with respect to $p$, set it to zero, and solve for $p$. However, it is often easier to work with the log-likelihood function and minimize the negative log-likelihood instead.

$$
\ell(p) = \log(L(p)) = \log\left(\binom{10}{7}\right) + 7 \log(p) + 3 \log(1-p)
$$

Taking the derivative of the log-likelihood function with respect to $p$ and setting it to zero gives:
$$
\frac{d\ell(p)}{dp} = \frac{7}{p} - \frac{3}{1-p} = 0
$$

Now we can solve for $p$:

$$
p = \frac{7}{10} = 0.7
$$

In other words the ML estimate for the probability of success is simply the observed proportion of successes in the data.

You can quickly check that every other value of $p$ gives a lower likelihood than $p=0.7$.



In [ ]:
# Probability of 7 successes in 10 trials for different p
prob_7_successes <- dbinom(7, size = 10, prob = 0.7)
cat(sprintf("Probability of exactly 7 successes in 10 trials with p=0.7: %.4f\n", prob_7_successes))

prob_7_successes <- dbinom(7, size = 10, prob = 0.6)
cat(sprintf("Probability of exactly 7 successes in 10 trials with p=0.6: %.4f\n", prob_7_successes))

prob_7_successes <- dbinom(7, size = 10, prob = 0.8)
cat(sprintf("Probability of exactly 7 successes in 10 trials with p=0.8: %.4f\n", prob_7_successes))
